# Lesson 10 — SaaS and System Dependency Rationalization

## Goal
Analyze software spend and system dependencies as PE value levers.

## Learning Objectives

1. Build a SaaS inventory with seat utilization and cost-per-active-user metrics
2. Detect duplicate tools in the same functional category using a scoring model
3. Build a system dependency graph that reveals which workflows depend on which tools
4. Identify manual bridges between systems and quantify their friction cost (using L06 methods)
5. Score and rank rationalization opportunities by annual saving, implementation risk, and effort

## Prerequisites

- Lesson 06: Cost of Friction Model (manual bridge cost uses friction EUR methodology)
- Lesson 08: Graph Bottlenecks to Value (system dependency graph uses same networkx approach)
- Lesson 09: Working Capital and Cash Value (SaaS rationalization can free budget for AI investment)

## Important Caution

> **Do not claim that every SaaS system can be replaced by an AI agent.**
>
> The value from SaaS rationalization usually comes from:
> - Reducing **seat growth** on underutilized tools
> - Removing **duplicate tools** in the same functional category
> - Automating **manual bridges** between systems
> - Deprecating **narrow point solutions** that solve one problem
> - Improving **utilization** of existing enterprise licenses
> - Reducing **process workarounds** created by system gaps

## Core Insight

**SaaS rationalization is not about replacing software with AI. It is about eliminating
the hidden cost of software sprawl: duplicate licenses, manual data transfers between
systems, and the cognitive overhead of too many tools.**
PE firms target SaaS as a direct EBITDA lever — every EUR of eliminated SaaS spend
flows directly to the bottom line.

---

## Part 1 — SaaS Rationalization Concepts

### The Five Rationalization Levers

| Lever | Description | Typical saving | AI role |
|-------|------------|---------------|---------|
| **Seat reduction** | Remove licenses for inactive or departed users | 15–30% of tool cost | Usage analytics, off-boarding automation |
| **Duplicate consolidation** | Merge two tools doing the same job | 40–60% of one tool | None — governance decision |
| **Manual bridge automation** | Replace copy-paste data transfers with integration | L06 friction EUR | Integration AI, ETL automation |
| **Point solution deprecation** | Retire narrow tools with <20% utilization | 100% of tool cost | Workflow AI replacing the function |
| **Utilization improvement** | Increase adoption of underused enterprise licenses | Avoids next seat purchase | Training, onboarding AI |

### Key Metrics

| Metric | Formula | Good / Warning |
|--------|---------|----------------|
| **Seat utilization** | Active users / licensed seats | >70% good; <40% at risk |
| **Cost per active user** | Annual cost / active users | Compare to market benchmark |
| **Overlap score** | Functions shared with another tool / total functions | >50% = duplicate candidate |
| **Manual bridge count** | Handoffs requiring copy-paste across systems | 0 = target |
| **Shadow IT count** | Tools used but not in approved inventory | >5 = governance risk |

### Connection to L06 / L08 / L09

- **L06**: Manual bridges are friction. `cases × copy_paste_minutes × hourly_cost = annual friction EUR`.
- **L08**: The system dependency graph uses the same networkx centrality approach as the process graph.
  A system with high betweenness centrality is a single point of failure — and a rationalization risk.
- **L09**: SaaS savings reduce annual OpEx. At an 8x EBITDA multiple, EUR 100k SaaS saving = EUR 800k enterprise value.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
print('Libraries loaded: pandas', pd.__version__, '| networkx', nx.__version__)

---

## Part 2 — SaaS Analysis Utilities

| Function | Purpose |
|----------|---------|
| `seat_utilization()` | Active users / licensed seats; flags tools below threshold |
| `cost_per_active_user()` | Annual cost divided by active user count |
| `duplicate_overlap_score()` | Shared functional categories / total categories for a pair |
| `manual_bridge_cost()` | Friction EUR from L06 method: cases × minutes × hourly rate |
| `rationalization_saving()` | Expected annual saving from each lever type |

In [ ]:
def seat_utilization(active_users, licensed_seats):
    return active_users / licensed_seats if licensed_seats > 0 else 0


def cost_per_active_user(annual_cost, active_users):
    return annual_cost / active_users if active_users > 0 else float('inf')


def duplicate_overlap_score(functions_a, functions_b):
    set_a, set_b = set(functions_a), set(functions_b)
    if not set_a or not set_b:
        return 0.0
    overlap = len(set_a & set_b)
    union   = len(set_a | set_b)
    return overlap / union


def manual_bridge_cost(annual_cases, minutes_per_transfer, loaded_hourly_cost):
    return annual_cases * (minutes_per_transfer / 60) * loaded_hourly_cost


def rationalization_saving(lever, annual_cost, utilization=None,
                           consolidation_pct=0.5, reduction_pct=0.2):
    if lever == 'seat_reduction':
        excess = max(0, 1.0 - (utilization or 0.5)) * annual_cost
        return excess * reduction_pct
    elif lever == 'duplicate_consolidation':
        return annual_cost * consolidation_pct
    elif lever == 'point_solution_deprecation':
        return annual_cost
    elif lever == 'utilization_improvement':
        return annual_cost * 0.10
    return 0.0


print('SaaS rationalization functions loaded.')
print('  seat_utilization()')
print('  cost_per_active_user()')
print('  duplicate_overlap_score()')
print('  manual_bridge_cost()')
print('  rationalization_saving()')

---

## Part 3 — SaaS Inventory: Building the Baseline

### Scenario

A 200-person PE-backed professional services company has accumulated 24 SaaS tools
across 5 years of growth. The CIO has asked for a rationalization scan before the
next budget cycle. The company has:

- No formal SaaS governance process
- 6 tools with <40% seat utilization
- At least 3 functional categories with duplicate tools
- Several manual data transfers (copy-paste between systems) that create L06-style friction

### SaaS Categories

| Category | Description | Expected tool count |
|----------|------------|---------------------|
| CRM | Customer relationship management | 1–2 |
| Collaboration | Messaging, video, shared docs | 2–3 |
| Project management | Task tracking, roadmaps | 2–3 |
| Finance | ERP, invoicing, expense | 2 |
| HR | HRIS, payroll, recruitment | 2 |
| Analytics | BI, dashboards, data viz | 2 |
| Document | Storage, wiki, knowledge base | 2–3 |
| Security | SSO, IAM, endpoint | 2 |
| Dev tools | CI/CD, monitoring, repo | 3 |

In [ ]:
saas_inventory = pd.DataFrame({
    'tool': [
        'Salesforce', 'HubSpot', 'Slack', 'Microsoft Teams', 'Jira',
        'Asana', 'Monday.com', 'NetSuite', 'QuickBooks', 'Workday',
        'BambooHR', 'Tableau', 'Power BI', 'Confluence', 'Notion',
        'SharePoint', 'Okta', 'CrowdStrike', 'GitHub', 'Datadog',
        'PagerDuty', 'Calendly', 'Loom', 'Zoom',
    ],
    'category': [
        'CRM', 'CRM', 'Collaboration', 'Collaboration', 'Project Mgmt',
        'Project Mgmt', 'Project Mgmt', 'Finance', 'Finance', 'HR',
        'HR', 'Analytics', 'Analytics', 'Document', 'Document',
        'Document', 'Security', 'Security', 'Dev Tools', 'Dev Tools',
        'Dev Tools', 'Productivity', 'Productivity', 'Collaboration',
    ],
    'licensed_seats': [
        200, 80, 200, 200, 120,
        60,  40,  50,  30, 200,
        200, 40,  200, 120, 80,
        200, 200, 200, 80,  40,
        30,  50,  60, 200,
    ],
    'active_users_30d': [
        155, 30,  188, 62,  108,
        22,  12,   48, 11,  178,
        35,  28,  165, 88,  65,
        70,  196, 198, 78,  38,
        18,  42,  25, 185,
    ],
    'annual_cost_eur': [
        96000, 14400, 72000, 36000, 43200,
        10800,  7200, 36000, 9600, 84000,
        22000, 28800, 14400, 21600, 9600,
        12000, 36000, 48000, 14400, 19200,
        12000,  3600,  4800, 36000,
    ],
    'owner_dept': [
        'Sales', 'Marketing', 'All', 'All', 'Engineering',
        'Operations', 'Marketing', 'Finance', 'Finance', 'HR',
        'HR', 'Analytics', 'All', 'Engineering', 'Operations',
        'All', 'IT', 'IT', 'Engineering', 'Engineering',
        'Engineering', 'Sales', 'All', 'All',
    ],
    'contract_renewal_months': [
        8, 2, 11, 5, 7,
        3, 1, 9, 1, 11,
        6, 4, 8, 7, 2,
        10, 9, 9, 6, 4,
        3, 1, 2, 5,
    ],
})

saas_inventory['utilization'] = saas_inventory.apply(
    lambda r: seat_utilization(r['active_users_30d'], r['licensed_seats']), axis=1
)
saas_inventory['cost_per_active'] = saas_inventory.apply(
    lambda r: cost_per_active_user(r['annual_cost_eur'], r['active_users_30d']), axis=1
)
saas_inventory['util_pct'] = (saas_inventory['utilization'] * 100).round(1)
saas_inventory['risk_flag'] = saas_inventory['utilization'].apply(
    lambda u: 'LOW UTIL' if u < 0.40 else ('WATCH' if u < 0.60 else 'OK')
)

total_saas_spend = saas_inventory['annual_cost_eur'].sum()
print('SAAS INVENTORY SUMMARY')
print('=' * 70)
print(f'Total tools:          {len(saas_inventory)}')
print(f'Total annual spend:   EUR {total_saas_spend:,.0f}')
print(f'Avg utilization:      {saas_inventory["utilization"].mean()*100:.1f}%')
print(f'Low utilization (<40%): {(saas_inventory["utilization"] < 0.4).sum()} tools')
print(f'Watch zone (40-60%):    {((saas_inventory["utilization"] >= 0.4) & (saas_inventory["utilization"] < 0.6)).sum()} tools')
print()
low_util = saas_inventory[saas_inventory['utilization'] < 0.4].sort_values('utilization')
print('LOW UTILIZATION TOOLS (< 40%):')
print(low_util[['tool','category','licensed_seats','active_users_30d','util_pct','annual_cost_eur','contract_renewal_months']].to_string(index=False))

In [ ]:
# Category-level analysis: spend and utilization by department
cat_summary = saas_inventory.groupby('category').agg(
    tool_count        = ('tool', 'count'),
    total_cost_eur    = ('annual_cost_eur', 'sum'),
    avg_utilization   = ('utilization', 'mean'),
    low_util_count    = ('risk_flag', lambda x: (x == 'LOW UTIL').sum()),
).reset_index()
cat_summary['avg_util_pct'] = (cat_summary['avg_utilization'] * 100).round(1)
cat_summary = cat_summary.sort_values('total_cost_eur', ascending=False)

print('SAAS SPEND AND UTILIZATION BY CATEGORY')
print('=' * 65)
print(cat_summary[['category','tool_count','total_cost_eur','avg_util_pct','low_util_count']].to_string(index=False))
print()

# Identify categories with >1 tool (duplicate candidates)
dup_cats = cat_summary[cat_summary['tool_count'] > 1]
print('DUPLICATE CANDIDATE CATEGORIES (>1 tool):')
for _, row in dup_cats.iterrows():
    tools_in_cat = saas_inventory[saas_inventory['category'] == row['category']]['tool'].tolist()
    spend_in_cat = row['total_cost_eur']
    print(f"  {row['category']}: {', '.join(tools_in_cat)} -- EUR {spend_in_cat:,.0f}/year total")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: spend by category (horizontal bars, color by avg util)
cat_plot = cat_summary.sort_values('total_cost_eur')
util_colors = ['#d73027' if u < 40 else '#fee08b' if u < 60 else '#4dac26'
               for u in cat_plot['avg_util_pct']]
axes[0].barh(cat_plot['category'], cat_plot['total_cost_eur'], color=util_colors, alpha=0.85)
axes[0].set_xlabel('Annual Cost (EUR)')
axes[0].set_title('SaaS Spend by Category\n(red=low util, yellow=watch, green=OK)')
axes[0].grid(axis='x', alpha=0.3)
for i, (cost, util) in enumerate(zip(cat_plot['total_cost_eur'], cat_plot['avg_util_pct'])):
    axes[0].text(cost + 200, i, f'{util:.0f}%', va='center', fontsize=8)

# Right: scatter — utilization vs annual cost per tool (bubble = licensed seats)
colors_r = ['#d73027' if u < 0.4 else '#fee08b' if u < 0.6 else '#4dac26'
            for u in saas_inventory['utilization']]
scatter = axes[1].scatter(
    saas_inventory['utilization'] * 100,
    saas_inventory['annual_cost_eur'],
    s=saas_inventory['licensed_seats'] * 1.5,
    c=colors_r, alpha=0.75, edgecolors='black', linewidth=0.8,
)
axes[1].axvline(40, color='#d73027', linestyle='--', linewidth=1.5, label='40% threshold')
axes[1].axvline(60, color='#e6a817', linestyle='--', linewidth=1.5, label='60% watch zone')
for _, row in saas_inventory[saas_inventory['utilization'] < 0.45].iterrows():
    axes[1].annotate(row['tool'], (row['utilization']*100, row['annual_cost_eur']),
                     textcoords='offset points', xytext=(5, 3), fontsize=7)
axes[1].set_xlabel('Seat Utilization (%)')
axes[1].set_ylabel('Annual Cost (EUR)')
axes[1].set_title('Utilization vs Cost\n(bubble size = licensed seats)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('saas_utilization.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Total SaaS spend: EUR {total_saas_spend:,.0f}/year across {len(saas_inventory)} tools')

---

## Part 4 — Duplicate Tool Detection and Consolidation Value

### How Duplicates Accumulate

Duplicate tools emerge when:

- Different departments procure independently (no central IT approval)
- A tool is adopted by one team; a second tool is adopted by another before a review
- An acquisition brings in a parallel stack
- A pilot tool is never deprecated after a full rollout

### Consolidation Decision Framework

For each duplicate pair, evaluate:

| Question | Guidance |
|----------|---------|
| Is one tool the strategic standard? | Prefer the tool with higher utilization and executive sponsor |
| Which has the lower migration cost? | Check data export/import, API availability, retraining time |
| Are both in active contracts? | Prioritise the one closest to renewal |
| Is there a true capability gap? | If yes, consolidation requires augmentation — do not force-fit |

### Important Caution (repeated)

> Consolidation value is ONLY realised when the duplicate contract is **actually cancelled**.
> A decision to consolidate without contract termination saves nothing.
> Plan the migration before the renewal date, not after.

In [ ]:
# Duplicate detection: tools in the same category, scored by consolidation potential
# Functions defined per tool (simplified: category is primary overlap signal)
tool_functions = {
    'Salesforce':       ['crm', 'pipeline', 'contact_mgmt', 'reporting', 'forecasting'],
    'HubSpot':          ['crm', 'pipeline', 'contact_mgmt', 'email_marketing', 'forms'],
    'Slack':            ['messaging', 'channels', 'file_share', 'integrations', 'search'],
    'Microsoft Teams':  ['messaging', 'video_calls', 'file_share', 'calendar', 'sharepoint'],
    'Zoom':             ['video_calls', 'webinars', 'recording', 'scheduling'],
    'Jira':             ['task_tracking', 'sprints', 'backlog', 'reporting', 'roadmap'],
    'Asana':            ['task_tracking', 'projects', 'timelines', 'reporting'],
    'Monday.com':       ['task_tracking', 'projects', 'timelines', 'dashboards'],
    'NetSuite':         ['erp', 'invoicing', 'financial_reporting', 'inventory', 'payroll'],
    'QuickBooks':       ['invoicing', 'expense_tracking', 'financial_reporting', 'payroll'],
    'Workday':          ['hris', 'payroll', 'benefits', 'talent_mgmt', 'reporting'],
    'BambooHR':         ['hris', 'onboarding', 'performance', 'time_tracking'],
    'Tableau':          ['dashboards', 'data_viz', 'reporting', 'analytics'],
    'Power BI':         ['dashboards', 'data_viz', 'reporting', 'excel_integration'],
    'Confluence':       ['wiki', 'documentation', 'knowledge_base', 'collaboration'],
    'Notion':           ['wiki', 'documentation', 'notes', 'databases', 'projects'],
    'SharePoint':       ['document_storage', 'wiki', 'collaboration', 'workflow'],
    'Okta':             ['sso', 'mfa', 'user_provisioning', 'access_mgmt'],
    'CrowdStrike':      ['endpoint_security', 'threat_detection', 'incident_response'],
    'GitHub':           ['version_control', 'ci_cd', 'code_review', 'project_mgmt'],
    'Datadog':          ['monitoring', 'logging', 'apm', 'dashboards', 'alerts'],
    'PagerDuty':        ['alerting', 'on_call', 'incident_mgmt'],
    'Calendly':         ['scheduling', 'calendar_integration'],
    'Loom':             ['video_recording', 'async_communication', 'screen_capture'],
}

# Find all same-category pairs and score overlap
dup_pairs = []
tools_list = saas_inventory['tool'].tolist()
for i in range(len(tools_list)):
    for j in range(i + 1, len(tools_list)):
        t1, t2 = tools_list[i], tools_list[j]
        cat1 = saas_inventory.loc[saas_inventory['tool'] == t1, 'category'].iloc[0]
        cat2 = saas_inventory.loc[saas_inventory['tool'] == t2, 'category'].iloc[0]
        if cat1 == cat2 and t1 in tool_functions and t2 in tool_functions:
            score = duplicate_overlap_score(tool_functions[t1], tool_functions[t2])
            cost1 = saas_inventory.loc[saas_inventory['tool'] == t1, 'annual_cost_eur'].iloc[0]
            cost2 = saas_inventory.loc[saas_inventory['tool'] == t2, 'annual_cost_eur'].iloc[0]
            util1 = saas_inventory.loc[saas_inventory['tool'] == t1, 'utilization'].iloc[0]
            util2 = saas_inventory.loc[saas_inventory['tool'] == t2, 'utilization'].iloc[0]
            retire_tool = t1 if util1 <= util2 else t2
            retire_cost = cost1 if util1 <= util2 else cost2
            dup_pairs.append({
                'tool_a': t1, 'tool_b': t2, 'category': cat1,
                'overlap_score': round(score, 2),
                'combined_cost': cost1 + cost2,
                'retire_candidate': retire_tool,
                'potential_saving': retire_cost,
            })

dup_df = pd.DataFrame(dup_pairs).sort_values('overlap_score', ascending=False)
high_overlap = dup_df[dup_df['overlap_score'] >= 0.30]
print('DUPLICATE TOOL PAIRS (overlap score >= 0.30):')
print('=' * 80)
print(high_overlap[['tool_a','tool_b','category','overlap_score','combined_cost','retire_candidate','potential_saving']].to_string(index=False))
print()
print(f'Total potential saving from consolidation: EUR {high_overlap["potential_saving"].sum():,.0f}/year')

In [ ]:
# Seat reduction savings: tools with <40% utilization
seat_opps = saas_inventory[saas_inventory['utilization'] < 0.40].copy()
seat_opps['excess_seats'] = seat_opps['licensed_seats'] - seat_opps['active_users_30d']
seat_opps['cost_per_seat'] = seat_opps['annual_cost_eur'] / seat_opps['licensed_seats']
seat_opps['seat_saving'] = seat_opps['excess_seats'] * seat_opps['cost_per_seat'] * 0.80

# Point solution deprecation: tools < 40% util AND annual cost < EUR 12,000
point_solutions = seat_opps[seat_opps['annual_cost_eur'] <= 12000]

# Consolidation savings from high-overlap pairs
consol_saving = high_overlap['potential_saving'].sum()

# Total rationalization opportunity
seat_saving_total = seat_opps[~seat_opps['tool'].isin(point_solutions['tool'])]['seat_saving'].sum()
point_saving_total = point_solutions['annual_cost_eur'].sum()
total_rat_saving = consol_saving + seat_saving_total + point_saving_total

print('RATIONALIZATION VALUE SUMMARY')
print('=' * 60)
print()
print(f'1. DUPLICATE CONSOLIDATION')
print(f'   Pairs identified: {len(high_overlap)}')
print(f'   Potential saving:  EUR {consol_saving:,.0f}/year')
print()
print(f'2. SEAT REDUCTION (tools >40% excess seats)')
print(seat_opps[~seat_opps['tool'].isin(point_solutions['tool'])][['tool','licensed_seats','active_users_30d','excess_seats','seat_saving']].to_string(index=False))
print(f'   Potential saving:  EUR {seat_saving_total:,.0f}/year')
print()
print(f'3. POINT SOLUTION DEPRECATION (<40% util, <EUR 12k/year)')
print(point_solutions[['tool','annual_cost_eur','util_pct','contract_renewal_months']].to_string(index=False))
print(f'   Potential saving:  EUR {point_saving_total:,.0f}/year')
print()
print(f'TOTAL RATIONALIZATION OPPORTUNITY: EUR {total_rat_saving:,.0f}/year')
print(f'  = {total_rat_saving / total_saas_spend * 100:.1f}% of current SaaS spend')
ebitda_multiple = 8
print(f'  At {ebitda_multiple}x EBITDA multiple: EUR {total_rat_saving * ebitda_multiple:,.0f} enterprise value')

In [ ]:
# System dependency graph: which workflows depend on which tools
# Edge = workflow USES system; weight = manual_transfer_minutes if data must be copied
workflow_deps = [
    ('Invoice_Approval',  'NetSuite',        0,  'automated_sync'),
    ('Invoice_Approval',  'Salesforce',       0,  'automated_sync'),
    ('Invoice_Approval',  'QuickBooks',       25, 'manual_export'),
    ('Support_Triage',    'Microsoft Teams',  0,  'automated_sync'),
    ('Support_Triage',    'Jira',             15, 'manual_copy'),
    ('Support_Triage',    'Confluence',        0,  'automated_sync'),
    ('Support_Triage',    'Slack',            0,  'automated_sync'),
    ('Mgmt_Reporting',    'Power BI',         0,  'automated_sync'),
    ('Mgmt_Reporting',    'NetSuite',          0,  'automated_sync'),
    ('Mgmt_Reporting',    'Tableau',           30, 'manual_export'),
    ('Mgmt_Reporting',    'Datadog',           20, 'manual_export'),
    ('HR_Onboarding',     'Workday',           0,  'automated_sync'),
    ('HR_Onboarding',     'Okta',              0,  'automated_sync'),
    ('HR_Onboarding',     'GitHub',            10, 'manual_ticket'),
    ('HR_Onboarding',     'BambooHR',          25, 'manual_copy'),
    ('Sales_Pipeline',    'Salesforce',         0,  'automated_sync'),
    ('Sales_Pipeline',    'HubSpot',           30, 'manual_export'),
    ('Sales_Pipeline',    'Asana',             20, 'manual_copy'),
]

G_saas = nx.DiGraph()
for wf, sys, mins, method in workflow_deps:
    G_saas.add_edge(wf, sys, transfer_minutes=mins, method=method,
                    is_manual=(method != 'automated_sync'))

# Betweenness centrality: which systems are most central to workflows?
btw = nx.betweenness_centrality(G_saas, normalized=True)
deg = nx.degree_centrality(G_saas)

centrality_df = pd.DataFrame({
    'node': list(btw.keys()),
    'betweenness': list(btw.values()),
    'degree': [deg[n] for n in btw.keys()],
    'node_type': ['workflow' if n in [w for w,_,_,_ in workflow_deps] else 'system' for n in btw.keys()],
}).sort_values('betweenness', ascending=False)

systems_only = centrality_df[centrality_df['node_type'] == 'system'].head(8)
print('SYSTEM DEPENDENCY GRAPH — Top Systems by Betweenness Centrality')
print('=' * 65)
print(systems_only[['node','betweenness','degree']].to_string(index=False))
print()
manual_edges = [(w,s,m,t) for w,s,m,t in workflow_deps if t != 'automated_sync']
print(f'Manual bridges detected: {len(manual_edges)}')
for wf, sys, mins, method in sorted(manual_edges, key=lambda x: x[2], reverse=True):
    print(f'  {wf} -> {sys}: {mins} min/transfer ({method})')

---

## Part 5 — Manual Bridges: The Hidden Cost of System Gaps

### What Is a Manual Bridge?

A manual bridge is any workflow step that requires a human to:

- Export data from one system as a CSV or spreadsheet
- Copy-paste records between two systems that lack an integration
- Re-enter data because two tools cannot communicate
- Maintain a spreadsheet as the 'source of truth' between two systems

**Manual bridges are invisible in SaaS spend reports** — they appear in headcount,
not software budgets. This is why SaaS rationalization and L06 friction analysis
must be done together.

### The Manual Bridge Cost Formula (from L06)

```
Annual bridge cost = annual_transfers × minutes_per_transfer / 60 × loaded_hourly_cost
```

### AI vs Integration

| Solution | When to use | Cost | Risk |
|----------|------------|------|------|
| Native integration (API) | Both systems support it | Low | Low |
| iPaaS (Zapier, Make, Boomi) | Quick automation needed | Medium | Low |
| AI extraction + routing | Unstructured data, no API | Medium | Medium |
| Custom ETL pipeline | Complex transformation needed | High | High |
| **Do nothing** | Transfer <1h/week, low error risk | Zero | Low |

> Automate the bridge only when the **friction cost exceeds integration cost within 12 months**.

In [ ]:
# Manual bridge cost quantification (L06 friction method)
# Assumption: each manual transfer happens once per workflow execution
# Workflow volumes from L06/L07 data (Invoice=1050/Q, Support=4500/Q, Reporting=13/Q)

bridge_data = pd.DataFrame({
    'workflow':        ['Invoice_Approval', 'Mgmt_Reporting', 'Sales_Pipeline', 'HR_Onboarding', 'Support_Triage', 'Mgmt_Reporting', 'Sales_Pipeline'],
    'from_system':     ['Invoice_Approval', 'Mgmt_Reporting', 'Sales_Pipeline', 'HR_Onboarding', 'Support_Triage', 'Mgmt_Reporting', 'Sales_Pipeline'],
    'to_system':       ['QuickBooks', 'Tableau', 'HubSpot', 'BambooHR', 'Jira', 'Datadog', 'Asana'],
    'transfer_method': ['manual_export', 'manual_export', 'manual_export', 'manual_copy', 'manual_copy', 'manual_export', 'manual_copy'],
    'minutes_each':    [25, 30, 30, 25, 15, 20, 20],
    'q1_transfers':    [1050, 13, 80, 12, 4500, 13, 80],
    'loaded_hourly':   [55, 65, 60, 50, 55, 65, 60],
})

bridge_data['annual_transfers'] = bridge_data['q1_transfers'] * 4
bridge_data['annual_cost_eur'] = bridge_data.apply(
    lambda r: manual_bridge_cost(r['annual_transfers'], r['minutes_each'], r['loaded_hourly']),
    axis=1,
)

total_bridge_cost = bridge_data['annual_cost_eur'].sum()

print('MANUAL BRIDGE COST ANALYSIS')
print('=' * 80)
print(bridge_data[['workflow','to_system','minutes_each','annual_transfers','annual_cost_eur']].to_string(index=False))
print()
print(f'Total annual manual bridge cost: EUR {total_bridge_cost:,.0f}')
print()
print('L06 FRICTION CONNECTION:')
print('  Manual bridges are a subset of total L06 friction cost.')
print('  They represent avoidable work caused by missing system integrations,')
print('  not inherent complexity in the workflow itself.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: horizontal bar chart of bridge costs
bd = bridge_data.sort_values('annual_cost_eur')
bar_labels = bd['workflow'] + ' -> ' + bd['to_system']
axes[0].barh(bar_labels, bd['annual_cost_eur'], color='#d73027', alpha=0.75)
axes[0].set_xlabel('Annual Cost (EUR)')
axes[0].set_title('Manual Bridge Costs by System Pair')
axes[0].grid(axis='x', alpha=0.3)
for i, cost in enumerate(bd['annual_cost_eur']):
    axes[0].text(cost + 100, i, f'EUR {cost:,.0f}', va='center', fontsize=8)

# Right: system dependency graph (networkx layout)
# Only show workflows + systems that have manual bridges
manual_nodes = set()
manual_edges_draw = []
for wf, sys, mins, method in workflow_deps:
    if method != 'automated_sync':
        manual_nodes.add(wf)
        manual_nodes.add(sys)
        manual_edges_draw.append((wf, sys, mins))

G_manual = nx.DiGraph()
for wf, sys, mins in manual_edges_draw:
    G_manual.add_edge(wf, sys, weight=mins)

pos = nx.spring_layout(G_manual, k=2.5, seed=42)
workflow_nodes = [n for n in G_manual.nodes() if 'Pipeline' in n or 'Triage' in n or 'Approval' in n or 'Reporting' in n or 'Onboarding' in n]
system_nodes   = [n for n in G_manual.nodes() if n not in workflow_nodes]

nx.draw_networkx_nodes(G_manual, pos, nodelist=workflow_nodes, node_color='#4dac26', node_size=700, ax=axes[1], alpha=0.85)
nx.draw_networkx_nodes(G_manual, pos, nodelist=system_nodes,   node_color='#d73027', node_size=700, ax=axes[1], alpha=0.85)
nx.draw_networkx_labels(G_manual, pos, font_size=8, ax=axes[1])
edge_weights = [G_manual[u][v]['weight'] for u, v in G_manual.edges()]
nx.draw_networkx_edges(G_manual, pos, ax=axes[1],
    width=[w / 10 for w in edge_weights],
    edge_color='#888888', arrows=True, arrowsize=20,)
edge_labels = {(u,v): f"{G_manual[u][v]['weight']}m" for u,v in G_manual.edges()}
nx.draw_networkx_edge_labels(G_manual, pos, edge_labels=edge_labels, font_size=7, ax=axes[1])
axes[1].set_title('Manual Bridge Network\n(green=workflow, red=system, width=transfer minutes)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('manual_bridge_network.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Total manual bridge cost: EUR {total_bridge_cost:,.0f}/year')
print(f'% of total SaaS spend: {total_bridge_cost / total_saas_spend * 100:.1f}%')

---

## Part 6 — Rationalization Opportunity Scorecard

### From Analysis to Action

We now have four value levers quantified:

| Lever | Source of saving | Risk | Speed |
|-------|----------------|------|-------|
| Seat reduction | Cancel excess licenses at renewal | Low | Fast (next renewal) |
| Point solution deprecation | Retire low-use, low-cost tools | Low | Fast |
| Duplicate consolidation | Migrate users, cancel one contract | Medium | Medium |
| Manual bridge automation | Integration / AI routing | Medium | Medium |

### Scoring Method

For each opportunity:

```
opportunity_score = annual_saving_eur / (implementation_effort * risk_factor)
```

Where:
- `implementation_effort` = 1 (trivial) to 5 (complex multi-month project)
- `risk_factor` = 1 (low) to 5 (high business disruption risk)

A higher score = more value for less effort and risk.

### Prioritisation Logic

1. **Do first:** High saving, low effort, low risk — quick wins
2. **Plan next quarter:** High saving, medium effort — need a project plan
3. **Evaluate carefully:** Medium saving, high effort — may not be worth it
4. **Defer or skip:** Low saving, high effort — not a rationalization win

In [ ]:
# Build the consolidated opportunity scorecard
# Each row = one actionable rationalization move

# --- individual seat reduction opportunities ---
seat_rows = []
for _, row in seat_opps.iterrows():
    if row['tool'] not in point_solutions['tool'].values:
        seat_rows.append({
            'opportunity': f"Reduce {row['tool']} seats",
            'lever': 'Seat Reduction',
            'annual_saving_eur': round(row['seat_saving']),
            'impl_effort': 1,
            'risk': 1,
            'renewal_months': row['contract_renewal_months'],
        })

# --- point solution deprecations ---
point_rows = []
for _, row in point_solutions.iterrows():
    point_rows.append({
        'opportunity': f"Deprecate {row['tool']}",
        'lever': 'Point Solution',
        'annual_saving_eur': row['annual_cost_eur'],
        'impl_effort': 2,
        'risk': 2,
        'renewal_months': row['contract_renewal_months'],
    })

# --- consolidation opportunities (top pairs only) ---
consol_rows = []
seen_cats = set()
for _, row in high_overlap.iterrows():
    cat = row['category']
    if cat not in seen_cats:
        seen_cats.add(cat)
        consol_rows.append({
            'opportunity': f"Consolidate {cat} ({row['retire_candidate']})",
            'lever': 'Duplicate Consolidation',
            'annual_saving_eur': round(row['potential_saving']),
            'impl_effort': 3,
            'risk': 3,
            'renewal_months': saas_inventory.loc[saas_inventory['tool'] == row['retire_candidate'], 'contract_renewal_months'].iloc[0],
        })

# --- manual bridge automations ---
bridge_rows = []
for _, row in bridge_data.nlargest(4, 'annual_cost_eur').iterrows():
    bridge_rows.append({
        'opportunity': f"Automate bridge: {row['workflow']}->{row['to_system']}",
        'lever': 'Bridge Automation',
        'annual_saving_eur': round(row['annual_cost_eur']),
        'impl_effort': 3,
        'risk': 2,
        'renewal_months': 0,
    })

scorecard = pd.DataFrame(seat_rows + point_rows + consol_rows + bridge_rows)
scorecard['opportunity_score'] = (scorecard['annual_saving_eur'] /
    (scorecard['impl_effort'] * scorecard['risk'])).round(0).astype(int)
scorecard['urgent'] = scorecard['renewal_months'] <= 3
scorecard = scorecard.sort_values('opportunity_score', ascending=False)

print('RATIONALIZATION OPPORTUNITY SCORECARD')
print('=' * 90)
pd.set_option('display.max_colwidth', 50)
print(scorecard[['opportunity','lever','annual_saving_eur','impl_effort','risk','renewal_months','opportunity_score','urgent']].to_string(index=False))
print()
print(f'TOTAL SCORECARD VALUE: EUR {scorecard["annual_saving_eur"].sum():,.0f}/year')
urgent_total = scorecard[scorecard['urgent']]['annual_saving_eur'].sum()
print(f'URGENT (renewal <= 3 months): EUR {urgent_total:,.0f}/year')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: quadrant chart — annual saving vs implementation effort
lever_colors = {
    'Seat Reduction':        '#4dac26',
    'Point Solution':        '#984ea3',
    'Duplicate Consolidation': '#ff7f00',
    'Bridge Automation':     '#377eb8',
}
for _, row in scorecard.iterrows():
    c = lever_colors.get(row['lever'], '#888')
    axes[0].scatter(row['impl_effort'], row['annual_saving_eur'] / 1000,
        s=150, c=c, alpha=0.8, edgecolors='black', linewidth=0.6)
    axes[0].annotate(row['opportunity'][:28], (row['impl_effort'], row['annual_saving_eur'] / 1000),
        textcoords='offset points', xytext=(5, 3), fontsize=6)
axes[0].axvline(2.5, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[0].axhline(scorecard['annual_saving_eur'].median() / 1000, color='gray', linestyle='--', linewidth=1, alpha=0.5)
axes[0].set_xlabel('Implementation Effort (1=easy, 5=hard)')
axes[0].set_ylabel('Annual Saving (EUR thousands)')
axes[0].set_title('Opportunity Quadrant\n(top-left = quick wins)')
axes[0].grid(alpha=0.3)
from matplotlib.patches import Patch
legend_elements = [Patch(fc=c, label=l) for l, c in lever_colors.items()]
axes[0].legend(handles=legend_elements, fontsize=7, loc='upper right')

# Right: stacked bar chart — total saving by lever type
lever_totals = scorecard.groupby('lever')['annual_saving_eur'].sum().sort_values(ascending=True)
colors_bar = [lever_colors.get(l, '#888') for l in lever_totals.index]
axes[1].barh(lever_totals.index, lever_totals.values, color=colors_bar, alpha=0.85)
axes[1].set_xlabel('Annual Saving (EUR)')
axes[1].set_title('Total Saving by Rationalization Lever')
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(lever_totals.values):
    axes[1].text(v + 200, i, f'EUR {v:,.0f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('rationalization_scorecard.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Total rationalization value: EUR {scorecard["annual_saving_eur"].sum():,.0f}/year')

---

## Exercise 1 — Guided: Seat Utilization Scan

A portfolio company has the following SaaS tools. Identify which to flag and calculate
the potential savings if excess seats are reduced to 110% of active users.

| Tool | Licensed | Active (30d) | Annual Cost |
|------|----------|-------------|-------------|
| DocuSign | 150 | 45 | EUR 18,000 |
| Salesforce | 200 | 160 | EUR 96,000 |
| Slack | 200 | 195 | EUR 72,000 |
| Miro | 100 | 28 | EUR 9,600 |
| HubSpot | 80 | 22 | EUR 14,400 |
| Zendesk | 50 | 38 | EUR 24,000 |

**Task:**

1. Calculate seat utilization for each tool
2. Flag any tool with utilization < 40% as LOW UTIL, 40-60% as WATCH
3. Calculate potential saving for LOW UTIL tools: reduce seats to 110% of active users
4. Identify the highest-value rationalization candidate
5. What should the PE analyst do about the tools closest to contract renewal?

In [ ]:
# Exercise 1 answer key
ex1 = pd.DataFrame({
    'tool':     ['DocuSign', 'Salesforce', 'Slack', 'Miro', 'HubSpot', 'Zendesk'],
    'licensed': [150, 200, 200, 100, 80, 50],
    'active':   [45,  160, 195, 28,  22, 38],
    'cost_eur': [18000, 96000, 72000, 9600, 14400, 24000],
})
ex1['utilization'] = ex1['active'] / ex1['licensed']
ex1['flag'] = ex1['utilization'].apply(
    lambda u: 'LOW UTIL' if u < 0.40 else ('WATCH' if u < 0.60 else 'OK')
)
ex1['cost_per_seat'] = ex1['cost_eur'] / ex1['licensed']
ex1['optimal_seats'] = (ex1['active'] * 1.10).astype(int)
ex1['excess_seats'] = (ex1['licensed'] - ex1['optimal_seats']).clip(lower=0)
ex1['potential_saving'] = ex1['excess_seats'] * ex1['cost_per_seat']

print('EXERCISE 1 ANSWER KEY')
print('=' * 75)
print(ex1[['tool','licensed','active','utilization','flag','excess_seats','potential_saving']].assign(
    utilization=lambda d: (d['utilization']*100).round(1),
    potential_saving=lambda d: d['potential_saving'].round(0)
).to_string(index=False))
print()
print(f'Total seat reduction saving: EUR {ex1["potential_saving"].sum():,.0f}')
low_util = ex1[ex1['flag'] == 'LOW UTIL'].sort_values('potential_saving', ascending=False)
print()
print('Insight: DocuSign (30%) and Miro (28%) and HubSpot (28%) are LOW UTIL.')
print('Highest value: DocuSign has EUR 15k+ in excess seats AND only 30% utilization.')
print('Action: Check DocuSign renewal date -- if < 3 months, trigger renegotiation NOW.')

---

## Exercise 2 — Open-Ended: Duplicate Consolidation Business Case

Your portfolio company runs both **Jira** (120 seats, EUR 43,200/year, 90% util)
and **Asana** (60 seats, EUR 10,800/year, 37% util).

The CTO proposes consolidating everyone to Jira. The CFO wants a business case.

**Assumptions for your model:**

- Migration effort: 3 sprints (6 weeks) of 1 FTE project manager at EUR 90,000/year
- Retraining: 22 Asana users × 2 hours × EUR 50/hr fully loaded
- Productivity dip: 22 users × 5% efficiency loss × 3 months × EUR 50,000 avg salary
- Asana data migration: EUR 2,500 (consultant day rate)
- Annual saving: full Asana contract + excess Jira seats (10 seats freed)

**Task:**

1. Calculate total one-time migration cost
2. Calculate annual saving
3. Calculate payback period (months)
4. Calculate 3-year NPV at 10% discount rate (assume saving is end-of-year)
5. Is this consolidation worth doing? What additional information would you want?

In [ ]:
# Exercise 2 answer key
# One-time migration costs
pm_cost      = 90000 / 12 * 1.5   # 6 weeks of 1 FTE PM
retraining   = 22 * 2 * 50
prod_dip     = 22 * 0.05 * (3/12) * 50000
migration    = 2500
total_migration_cost = pm_cost + retraining + prod_dip + migration

# Annual savings
asana_saving     = 10800
jira_cost_per    = 43200 / 120
jira_seat_saving = 10 * jira_cost_per
annual_saving    = asana_saving + jira_seat_saving

# Payback
payback_months = total_migration_cost / (annual_saving / 12)

# 3-year NPV at 10% discount rate (end-of-year cash flows)
r = 0.10
npv = sum([annual_saving / (1 + r) ** t for t in range(1, 4)]) - total_migration_cost

print('EXERCISE 2 ANSWER KEY')
print('=' * 60)
print()
print('ONE-TIME MIGRATION COSTS:')
print(f'  PM time (6 weeks):         EUR {pm_cost:,.0f}')
print(f'  Retraining (22 users):     EUR {retraining:,.0f}')
print(f'  Productivity dip:          EUR {prod_dip:,.0f}')
print(f'  Data migration:            EUR {migration:,.0f}')
print(f'  TOTAL MIGRATION COST:      EUR {total_migration_cost:,.0f}')
print()
print('ANNUAL SAVING:')
print(f'  Asana contract cancelled:  EUR {asana_saving:,.0f}')
print(f'  Jira seat reduction (10):  EUR {jira_seat_saving:,.0f}')
print(f'  TOTAL ANNUAL SAVING:       EUR {annual_saving:,.0f}')
print()
print(f'Payback period: {payback_months:.1f} months')
print(f'3-year NPV (@10%): EUR {npv:,.0f}')
print()
print('VERDICT: Positive NPV. Payback well under 24 months = typical PE hurdle.')
print('WANT TO KNOW: renewal date (most critical), integration dependencies, Jira admin capacity.')

---

## Exercise 3 — Capstone: Full SaaS Rationalization Business Case

Using the 24-tool inventory from Part 3, build a full PE-ready rationalization business case.

**Deliverables (combine into a single output):**

### A. Total Annual Saving

Combine all three levers:
1. Seat reduction: tools where utilization < 40%, reduce to 110% of active users
2. Duplicate consolidation: use the top 4 highest-overlap pairs
3. Manual bridge automation: model the 3 highest-cost bridges as eliminated

### B. Implementation Cost Estimate

- Seat reduction: EUR 0 (just renegotiate at renewal — no migration needed)
- Consolidation: EUR 8,000 per tool deprecated (migration, retraining, project)
- Bridge automation: EUR 5,000 per bridge eliminated (iPaaS setup + testing)

### C. ROI and Payback

- Calculate total implementation cost
- Calculate 12-month ROI = (first-year saving - impl cost) / impl cost
- Calculate payback period in months

### D. PE Investment Committee Summary (3 bullet points)

Write three bullet points suitable for a PE IC memo. Use EUR numbers, not percentages.

> Hint: Use `rationalization_saving()` from Part 2 as your final aggregation function.

In [ ]:
# Exercise 3 capstone answer key

# A. Annual savings
# 1. Seat reduction (LOW UTIL tools only)
seat_ex3 = saas_inventory[saas_inventory['utilization'] < 0.40].copy()
seat_ex3['optimal_seats'] = (seat_ex3['active_users_30d'] * 1.10).astype(int)
seat_ex3['excess_seats'] = seat_ex3['licensed_seats'] - seat_ex3['optimal_seats']
seat_ex3['cost_per_seat'] = seat_ex3['annual_cost_eur'] / seat_ex3['licensed_seats']
seat_ex3['saving'] = seat_ex3['excess_seats'] * seat_ex3['cost_per_seat']
seat_saving_ex3 = seat_ex3['saving'].sum()

# 2. Duplicate consolidation (top 4 pairs by overlap score)
top4_pairs = high_overlap.head(4)
consol_saving_ex3 = top4_pairs['potential_saving'].sum()
tools_deprecated = len(top4_pairs)

# 3. Manual bridge automation (top 3 by cost)
top3_bridges = bridge_data.nlargest(3, 'annual_cost_eur')
bridge_saving_ex3 = top3_bridges['annual_cost_eur'].sum()
bridges_automated = len(top3_bridges)

# B. Implementation costs
seat_impl_cost   = 0
consol_impl_cost = tools_deprecated * 8000
bridge_impl_cost = bridges_automated * 5000
total_impl_cost  = seat_impl_cost + consol_impl_cost + bridge_impl_cost

# C. ROI and payback
total_annual_saving = rationalization_saving(seat_saving_ex3, consol_saving_ex3, bridge_saving_ex3)
first_year_saving = total_annual_saving - total_impl_cost
roi_12m = first_year_saving / total_impl_cost if total_impl_cost > 0 else float('inf')
payback_m = (total_impl_cost / (total_annual_saving / 12)) if total_annual_saving > 0 else float('inf')

print('EXERCISE 3 CAPSTONE ANSWER KEY')
print('=' * 65)
print()
print('A. ANNUAL SAVINGS:')
print(f'   Seat reduction:          EUR {seat_saving_ex3:,.0f}')
print(f'   Duplicate consolidation: EUR {consol_saving_ex3:,.0f}')
print(f'   Manual bridge automation:EUR {bridge_saving_ex3:,.0f}')
print(f'   TOTAL ANNUAL SAVING:     EUR {total_annual_saving:,.0f}')
print()
print('B. IMPLEMENTATION COSTS:')
print(f'   Seat renegotiation:      EUR {seat_impl_cost:,.0f} (zero -- renewal only)')
print(f'   Tool consolidation:      EUR {consol_impl_cost:,.0f} ({tools_deprecated} tools x EUR 8k)')
print(f'   Bridge automation:       EUR {bridge_impl_cost:,.0f} ({bridges_automated} bridges x EUR 5k)')
print(f'   TOTAL IMPL COST:         EUR {total_impl_cost:,.0f}')
print()
print('C. ROI AND PAYBACK:')
print(f'   12-month ROI:            {roi_12m*100:.0f}%')
print(f'   Payback period:          {payback_m:.1f} months')
print()
print('D. PE IC MEMO BULLETS:')
print(f'  - SaaS rationalization plan yields EUR {total_annual_saving:,.0f}/year recurring saving')
print(f'    through seat rightsizing, {tools_deprecated} tool consolidations, and {bridges_automated} integration automations.')
print(f'  - Total implementation cost of EUR {total_impl_cost:,.0f} delivers 12-month ROI of')
print(f'    {roi_12m*100:.0f}% with {payback_m:.0f}-month payback — within standard 100-day plan horizon.')
ebitda_multiple = 8
print(f'  - At {ebitda_multiple}x EBITDA, annual saving equates to EUR {total_annual_saving * ebitda_multiple:,.0f}')
print(f'    in enterprise value creation with no headcount reduction required.')

---

## Part 8 — Completion Checklist and Key Insights

### Lesson 10 Checklist

- [ ] I can calculate seat utilization and cost-per-active-user for a SaaS tool
- [ ] I can identify duplicate tool categories and score functional overlap
- [ ] I can model the three rationalization levers (seat, consolidation, bridge)
- [ ] I understand how system dependency graphs reveal rationalization risk
- [ ] I can build an opportunity scorecard with effort/risk scoring
- [ ] I can write a PE IC-ready SaaS rationalization summary with EUR numbers

### Key Insights from Lesson 10

1. **Utilization is the first signal, not the final verdict.**
   Low utilization may mean the tool is redundant, or may mean users were never onboarded.
   Validate with the tool owner before recommending cancellation.

2. **Duplicate tools cost more than their combined contract value.**
   They also create manual bridges, split institutional knowledge, and increase IT surface area.
   The true cost of duplication includes operational friction.

3. **Manual bridges are headcount costs disguised as process.**
   They appear in salary lines, not SaaS spend reports.
   Rationalizing the tool stack without addressing manual bridges leaves value on the table.

4. **System dependency graphs reveal rationalization risk.**
   High-betweenness tools that are also low-utilization are the most dangerous to retire:
   other workflows depend on them even if few users actively log in.

5. **Do not claim AI will replace every SaaS tool.**
   AI can automate specific workflows within a tool, but replacing an ERP or HRIS
   with an AI agent is a multi-year re-implementation project — not a rationalization win.

6. **Contract renewal timing is the key execution variable.**
   The best business case means nothing if the renewal was signed last month.
   SaaS rationalization must be linked to the contract calendar.

### Next Lesson

**L11 — AI Opportunity Scoring and Prioritisation Framework**

We will move from individual case analysis to a portfolio-level scoring model:
- How to systematically prioritise AI and automation opportunities across a portfolio company
- The AI opportunity scorecard: feasibility × value × risk × speed
- How to sequence implementations to maximise early wins and EBITDA impact
- Building the 100-day AI value plan from the analyses in L06–L10